# 03 - Build Training Datasets

## Stage 4 Objective

Build training-ready CSV files without requiring user-provided datasets.

By default, this notebook downloads the public LexGLUE LEDGAR dataset from Hugging Face:

- `data/processed/classification_dataset.csv`: clause type labels from LEDGAR.
- `data/processed/simplification_dataset.csv`: public legal clauses with weak auto-generated plain-language targets.

The simplification targets are weak targets, not expert-written simplifications. They make the training pipeline runnable, but human-written targets are still better for final model quality.

## 1. Configure Paths, Imports, and Public Dataset Limits

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset_builder import (
    PUBLIC_DATASET_CONFIG,
    PUBLIC_DATASET_NAME,
    assign_splits,
    build_public_ledgar_datasets,
    class_distribution,
    find_missing_values,
    get_classification_columns,
    get_simplification_columns,
    split_distribution,
)

SIMPLIFICATION_PATH = PROJECT_ROOT / 'data' / 'processed' / 'simplification_dataset.csv'
CLASSIFICATION_PATH = PROJECT_ROOT / 'data' / 'processed' / 'classification_dataset.csv'

MAX_TRAIN_ROWS = 600
MAX_VALIDATION_ROWS = 100
MAX_TEST_ROWS = 100
MAX_CLAUSE_TYPES = 10
SEED = 42

print(f'Public dataset: {PUBLIC_DATASET_NAME} / {PUBLIC_DATASET_CONFIG}')
print(f'Simplification output: {SIMPLIFICATION_PATH}')
print(f'Classification output: {CLASSIFICATION_PATH}')

## 2. Load Public LEDGAR Data

The notebook samples public data to stay Colab-friendly. Increase the row limits after the pipeline runs successfully.

In [ ]:
source_description = 'huggingface_ledgar'

simplification_rows, classification_rows = build_public_ledgar_datasets(
    max_train_rows=MAX_TRAIN_ROWS,
    max_validation_rows=MAX_VALIDATION_ROWS,
    max_test_rows=MAX_TEST_ROWS,
    max_clause_types=MAX_CLAUSE_TYPES,
    seed=SEED,
)

simplification_df = pd.DataFrame(simplification_rows, columns=get_simplification_columns())
classification_df = pd.DataFrame(classification_rows, columns=get_classification_columns())

if simplification_df.empty or classification_df.empty:
    raise ValueError(
        'No training rows were created. Check Hugging Face dataset access and row limits.'
    )

expected_splits = {'train', 'validation', 'test'}
if not expected_splits.issubset(set(classification_df['split'])):
    print('Reassigning splits so train, validation, and test are all present after sampling/filtering.')
    reassigned_splits = assign_splits(
        classification_df.to_dict(orient='records'),
        seed=SEED,
        stratify_key='clause_type',
    )
    classification_df['split'] = reassigned_splits
    split_by_clause_id = dict(zip(classification_df['clause_id'], classification_df['split']))
    simplification_df['split'] = simplification_df['clause_id'].map(split_by_clause_id).fillna(simplification_df['split'])
    classification_rows = classification_df.to_dict(orient='records')
    simplification_rows = simplification_df.to_dict(orient='records')

print(f'Dataset source: {source_description}')
print(f'Simplification rows: {len(simplification_df)}')
print(f'Classification rows: {len(classification_df)}')
display(simplification_df.head())
display(classification_df.head())

## 3. Check Missing Labels and Targets

In [ ]:
simplification_missing = find_missing_values(
    simplification_rows,
    ['clause_id', 'clause_text', 'simple_clause', 'split'],
)
classification_missing = find_missing_values(
    classification_rows,
    ['clause_id', 'clause_text', 'clause_type', 'risk_level', 'risk_type', 'split'],
)

print('Simplification missing values:')
display(pd.DataFrame([simplification_missing]))
print('Classification missing values:')
display(pd.DataFrame([classification_missing]))

if simplification_missing.get('simple_clause', 0) > 0:
    raise ValueError('Simplification dataset contains missing simple_clause targets.')
if classification_missing.get('clause_type', 0) > 0:
    raise ValueError('Classification dataset contains missing clause_type labels.')

## 4. Review Class Distribution

In [ ]:
for column in ['clause_type', 'risk_level', 'risk_type']:
    distribution = class_distribution(classification_rows, column)
    print(f'{column} distribution')
    display(pd.DataFrame(distribution.items(), columns=[column, 'count']).sort_values('count', ascending=False).head(20))

## 5. Review Train, Validation, and Test Splits

In [ ]:
print('Simplification split distribution')
display(pd.DataFrame(split_distribution(simplification_rows).items(), columns=['split', 'count']))

print('Classification split distribution')
display(pd.DataFrame(split_distribution(classification_rows).items(), columns=['split', 'count']))

missing_splits = {'train', 'validation', 'test'} - set(classification_df['split'])
if missing_splits:
    print(f'Warning: classification dataset is missing splits: {sorted(missing_splits)}')

## 6. Save Datasets

In [ ]:
SIMPLIFICATION_PATH.parent.mkdir(parents=True, exist_ok=True)
simplification_df.to_csv(SIMPLIFICATION_PATH, index=False)
classification_df.to_csv(CLASSIFICATION_PATH, index=False)

print(f'Saved {len(simplification_df)} row(s) to {SIMPLIFICATION_PATH.relative_to(PROJECT_ROOT)}')
print(f'Saved {len(classification_df)} row(s) to {CLASSIFICATION_PATH.relative_to(PROJECT_ROOT)}')